In [1]:
# =============================================================================
# CELL 1: Imports, Configuration, and Path Setup
# =============================================================================

import os
import sys
import json
import csv
import random
from pathlib import Path
import subprocess  # used for hf to gguf conversion

# 1. Environment Variables for HPC
# Ensure all caches and temp files go to scratch, not home directory
SCRATCH_ROOT = Path("/scratch/cpowel21/vllm_tuning")
os.environ['HF_HOME'] = str(SCRATCH_ROOT / ".hf_cache")
os.environ['TRANSFORMERS_CACHE'] = str(SCRATCH_ROOT / ".hf_cache")
os.environ['HF_DATASETS_CACHE'] = str(SCRATCH_ROOT / ".hf_datasets_cache")
os.environ['PIP_CACHE_DIR'] = str(SCRATCH_ROOT / ".pip_cache")
os.environ['TMPDIR'] = str(SCRATCH_ROOT / "tmp")

# Create directories if they don't exist
for d in [os.environ['HF_HOME'], os.environ['TMPDIR']]:
    Path(d).mkdir(parents=True, exist_ok=True)

# 2. Core Libraries
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # addressed transient vram ooms
import torch
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm

# 3. Unsloth & Vision Support
import unsloth
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

# 4. Training Framework (TRL)
from trl import SFTConfig, SFTTrainer

# 5. Datasets
import datasets

# 6. Print Environment Status
print(f"Python Path: {sys.executable}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Unsloth Version: {unsloth.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.2f} GB")

# =============================================================================
# CONFIGURATION: Paths and Hyperparameters
# Modify these variables to tune your training run
# =============================================================================

# --- Paths ---
BASE_MODEL_PATH = SCRATCH_ROOT / "models" / "qwen3-vl-4b-instruct"
TEMPLATE_PATH = SCRATCH_ROOT / "herbarium_template.jinja"
DATA_ROOT = SCRATCH_ROOT / "data"          # images referenced RELATIVE to this
DATA_DIR  = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "val"
TEST_DIR  = DATA_ROOT / "test"
OUTPUT_DIR = SCRATCH_ROOT / "checkpoints"
# --- Training Hyperparameters ---
# Full Fine-Tuning Settings

BATCH_SIZE = 6           # Per device batch size. Keep low for full FT to manage VRAM.
GRAD_ACCUM = 2           # Gradient accumulation steps. Effective BS = BATCH_SIZE * GRAD_ACCUM * GPUs
MAX_SEQ_LEN = 6144       # Max sequence length for images/text
WEIGHT_DECAY = 0.01      # Weight decay
OPTIM = "adamw_torch_fused"
SCHEDULER_TYPE = "cosine"

P1_LEARNING_RATE = 3e-5     # Learning rate for Full FT (typically lower than LoRA)
P1_NUM_EPOCHS = 2           # Number of epochs
P1_WARMUP_STEPS = 10        # Warmup steps
P2_LEARNING_RATE = 2e-5     # Learning rate for Full FT (typically lower than LoRA)
P2_NUM_EPOCHS = 3           # Number of epochs
P2_WARMUP_STEPS = 15        # Warmup steps
# --- Precision Settings ---
LOAD_IN_4BIT = False     # Set to False for full 16-bit precision
DTYPE = torch.bfloat16   # Use bfloat16 for high precision on A100

# --- Logging ---
LOGGING_DIR = SCRATCH_ROOT / "logs"
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

CAPTION_PROMPT = r"""You are a herbarium specimen analyzer. Given a photo of a pressed herbarium sheet and the taxonomic name, examine the image and output EXACTLY one JSON object matching the schema below. Use ONLY the enumerated values listed for each field. Do NOT invent values. Do NOT output free text. If you cannot determine a field from the image, use its designated fallback value. Ensure terms selected are appropriate to the taxa.

OUTPUT FORMAT: one JSON object with four top-level keys: "type", "attached_photo", "structures", "refs".

---

FIELD DEFINITIONS

type (string — required value):
  "PH" — herbarium sheet photo (pressed specimen on a mounting sheet)
  "PI" — in-situ photo (living plant in its natural environment)
  Choose based on what the image shows.

attached_photo (boolean):
  true — a separate printed or attached photograph is visible on the sheet
  false — no attached photograph is visible
  Inspect the sheet for a second, distinct photograph different from the main specimen photo.

structures.foliage (string — required value):
  "present" — any leaves or needle-like structures are visible on the sheet
  "absent" — no foliage is visible
  Look for leaf blades, needles, or other leaf-like organs on the pressed material.

structures.foliage_type (string — required value):
  "leaf" — broad, flat leaf blade
  "needle" — narrow, cylindrical leaf
  "scale" — small, flat, overlapping leaf
  "frond" — compound fern or palm leaf
  "spine" — modified leaf spine/thorn
  "mixed" — multiple leaf types on the same specimen
  "unknown" — foliage present but type cannot be determined
  "none" — no foliage visible (use when foliage="absent")
  Choose the dominant leaf type if multiple are present. Use "mixed" only when distinctly different types are clearly visible together.

structures.stem (string — required value):
  "woody" — hard, lignified stem tissue
  "herbaceous" — soft, non-woody stem tissue
  "unknown" — stem nature cannot be determined from the image (also use when no stem is visible)
  On pressed specimens, woody stems may appear darker, thicker, or more rigid; herbaceous stems are typically thinner, greener, or pliable-looking.

structures.phenology (object — all six boolean fields required):
  ** REPRODUCTIVE TRAITS ONLY: this field captures reproductive developmental
     state only. Vegetative structures (leaf buds, vegetative shoot apices) do
     NOT count. **

  ** PRESSED-STATE CALIBRATION: all herbarium sheets are pressed, dried plant
     material. Reproductive structures are routinely desiccated, flattened,
     shrivelled, or blackened, and may sit as discrete clusters at leaf axils,
     nodes, or shoot apices. Judge each flag by ORGAN MORPHOLOGY AND POSITION
     (a distinct clustered/stalked structure separate from the leaf blade), NOT
     by freshness or colour. A structurally identifiable dried flower, fruit, or
     cone counts even when brown or black. Do not default to all-false merely
     because material is dark or withered — but do not promote diffuse necrotic
     leaf patches, blotches, or rot, which lack discrete reproductive form. **

  Evaluate each flag INDEPENDENTLY against what is visible on the specimen.
  Multiple flags may be true on the same specimen (e.g. a sheet bearing both
  open flowers and fruit -> flower=true AND fruit=true). If NO reproductive
  structure of any kind is visible, ALL six flags are false.

  "flower" (boolean):
    true — any of: closed flower buds; open flowers with visible petals,
           stamens, or pollen; or a flower cluster / inflorescence.
    false — none of the above visible.
  "fruit" (boolean):
    true — fruit at any stage (developing/unripe, mature/ripe, or
           dehiscent/splitting), OR bare seeds with the fruit removed,
           abscised, or otherwise not present.
    false — none of the above visible.
  "pollen_cone" (boolean):
    true — a male (pollen) cone / microsporangiate strobilus is visible
           (small strobili typical of conifers; active pollen not required).
    false — not visible.
  "seed_cone" (boolean):
    true — a female (seed) cone / megasporangiate strobilus is visible, at any
           maturity (immature open cone or mature cone with visible scales).
    false — not visible.
  "sporulating" (boolean):
    true — VISIBLE spore-bearing reproductive structures on a non-seed plant:
           fern sori/sporangia, lycophyte or horsetail strobili, or
           moss/liverwort sporophytes (setae and capsules).
    false — not visible (e.g. a sterile frond or shoot).
  "reproductive_unknown" (boolean):
    true — a reproductive structure is clearly present but cannot be confidently
           assigned to any of the categories above.
    false — otherwise.

  PHENOLOGY RULES:
  - Judge phenology solely from reproductive structures visible on the specimen.
    Disregard any textual claim of phenological state on the sheet.
  - The flags are INDEPENDENT. Do not force a single choice; set every flag that
    applies and leave the rest false.
  - Do not map spore structures onto flower/fruit/cone flags, and do not map
    cones onto flower/fruit flags. Use the dedicated flag.
  - For non-seed plants (ferns, lycophytes, horsetails, mosses, liverworts):
    use "sporulating" when spore-bearing structures are visible; all flags false
    for a sterile frond or shoot.
  - Set reproductive_unknown=true ONLY when a structure is clearly reproductive
    but unclassifiable. If it also fits a specific flag, set that flag instead
    and leave reproductive_unknown=false.
  - Do not guess at STAGE, but do not under-call PRESENCE. Three-way routing
    for a dried structure:
      (a) if it is a discrete reproductive organ AND you can identify its kind,
          set the specific flag (flower / fruit / pollen_cone / seed_cone /
          sporulating);
      (b) if it is clearly a discrete reproductive organ by position and form
          (e.g. a stalked or clustered structure at a leaf axil, node, or shoot
          apex) but desiccation/blackening prevents identifying its kind, set
          reproductive_unknown=true and leave the specific flags false;
      (c) if it is only a faint scrap, fragment, blotch, or necrotic/rotted leaf
          patch lacking discrete reproductive form, leave all flags false.
    Desiccation, flattening, or blackening alone is never grounds for (c).
    If ANY structure on the sheet is resolvable under (a), prefer (a) over (b).

refs.label (boolean):
  true — a mounted herbarium specimen label is visible bearing a scientific (taxon) name and/or
         collection data (locality, collection date, or collector). An institutional
         name, accession/barcode number, ruler caption, or color-chart text does NOT
         count, even when printed on a white slip.
  false — no specimen label is visible

refs.barcode (boolean):
  true — a striped (1D) or 2D barcode is visible
  false — no barcode is visible

refs.stamp (boolean):
  true — an ink stamp, seal, or institutional mark is visible
  false — no stamp is visible

refs.crc (boolean):
  true — a color reference chart (CRC chart) is visible
  false — no CRC chart is visible

refs.scale_bar (boolean):
  true — a ruler, scale bar, or measurement reference is visible
  false — no scale reference is visible

---

DECISION GUIDANCE

- Use "unknown" when the image quality, specimen condition, or viewing angle prevents reliable identification (applies to type, foliage_type, and stem).
- Use "none" only for foliage_type (when foliage="absent").
- Phenology is a set of independent boolean flags, not a single choice. Set every flag that applies and leave the rest false; when no reproductive structure is visible, all phenology flags are false. Do not guess — if a structure is too faint or ambiguous to call confidently, leave its flag false.
- Each ref.* field is independent — evaluate each one separately against what is visible.
- Each phenology flag is independent — evaluate each one separately against what is visible.

---
OUTPUT INSTRUCTIONS

Output ONLY a single JSON object. No markdown fences. No explanatory text. No additional keys. Each value below shows the TYPE and allowed range, not a default — decide every field from the image. Match this exact structure:

{
  "type": "<PH|PI>",
  "attached_photo": <true|false>,
  "structures": {
    "foliage": "<present|absent>",
    "foliage_type": "<leaf|needle|scale|frond|spine|mixed|unknown|none>",
    "stem": "<woody|herbaceous|unknown>",
    "phenology": {
      "flower": <true|false>,
      "fruit": <true|false>,
      "pollen_cone": <true|false>,
      "seed_cone": <true|false>,
      "sporulating": <true|false>,
      "reproductive_unknown": <true|false>
    }
  },
  "refs": {
    "label": <true|false>,
    "barcode": <true|false>,
    "stamp": <true|false>,
    "crc": <true|false>,
    "scale_bar": <true|false>
  }
}

Replace every <...> with one allowed value. Output only the JSON.
Use only the enumerated values defined above. Output nothing else."""

print(f"""
Configuration Summary:
----------------------
Model Path      : {BASE_MODEL_PATH}
Data Root       : {DATA_ROOT}
  Train / Val / Test : {DATA_DIR.name} / {VAL_DIR.name} / {TEST_DIR.name}
Output Path     : {OUTPUT_DIR}
Batch Size      : {BATCH_SIZE} (Effective: {BATCH_SIZE * GRAD_ACCUM})
Sequence Length : {MAX_SEQ_LEN}
Learning Rates  : P1:{P1_LEARNING_RATE} P2:{P2_LEARNING_RATE}
Epochs          : P1:{P1_NUM_EPOCHS} P2:{P2_NUM_EPOCHS}
Precision      :  {'BF16' if DTYPE == torch.bfloat16 else 'FP32'}
    """)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Python Path: /scratch/cpowel21/vllm_tuning/conda_envs/vllm_tuning_mamba/bin/python
PyTorch Version: 2.10.0+cu128
Unsloth Version: 2026.5.8
CUDA Available: True
GPU Name: NVIDIA A100-SXM4-80GB
GPU Memory: 85.09 GB

Configuration Summary:
----------------------
Model Path      : /scratch/cpowel21/vllm_tuning/models/qwen3-vl-4b-instruct
Data Root       : /scratch/cpowel21/vllm_tuning/data
  Train / Val / Test : train / val / test
Output Path     : /scratch/cpowel21/vllm_tuning/checkpoints
Batch Size      : 6 (Effective: 12)
Sequence Length : 6144
Learning Rates  : P1:3e-05 P2:2e-05
Epochs          : P1:2 P2:3
Precision      :  BF16
    


In [2]:
# =============================================================================
# CELL 2: Load Model (Full FT), and Prepare Datasets
# =============================================================================

# 1. Load Base Model with Full Fine-Tuning Enabled
#print("Loading base model in BF16 with Full Fine-Tuning...")
# model, tokenizer = FastVisionModel.from_pretrained(
#     model_name      = str(BASE_MODEL_PATH),
#     max_seq_length  = MAX_SEQ_LEN,
#     load_in_4bit    = LOAD_IN_4BIT,  # False for full tuning
#     dtype           = DTYPE,
#     full_finetuning = True,
# )

# # 2. Apply Chat Template
# print("Loading chat template...")
# if TEMPLATE_PATH.exists():
#     tokenizer.chat_template = TEMPLATE_PATH.read_text()
#     print(f"Loaded template from: {TEMPLATE_PATH}")
# else:
#     print(f"Warning: Template not found at {TEMPLATE_PATH}. Using default.")

# # 3. Verify Full Fine-Tuning Status
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total_params = sum(p.numel() for p in model.parameters())
# print(f"   Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

# 4. Define Formatters for Phase 1 & 2
# Phase 1: system prompt + image + taxon + instruction.
# Phase 2: image + taxon only (no system prompt) — streamlined inference format.
def format_phase1(sample: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": CAPTION_PROMPT},
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text",  "text": f"Taxon: {sample['taxon']}\nCaption this specimen image."},
            ]},
            {"role": "assistant", "content": sample["caption_json"]},  # bare JSON
        ]
    }

def format_phase2(sample: dict) -> dict:
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text",  "text": f"Taxon: {sample['taxon']}"},
            ]},
            {"role": "assistant", "content": sample["caption_json"]},  # bare JSON
        ]
    }

# 5. In-Memory Dataset Class
# Augmentation pipeline for training — image only, labels are unaffected
TRAIN_TRANSFORMS = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2)])

def taxon_from_path(rel_path: str) -> str:
    """images/<Taxon_Name>/<id>.jpg -> 'Taxon Name'."""
    return Path(rel_path).parent.name.replace("_", " ")

class InMemoryDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, formatter, shuffle: bool = False,
                 augment: bool = False):
        self.formatter  = formatter
        self.transforms = TRAIN_TRANSFORMS if augment else None

        print(f"Pre-loading {len(hf_dataset)} images into RAM...", flush=True)
        self.samples = []

        for row in hf_dataset:
            try:
                img     = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
                cap_str = json.dumps(json.loads(row["caption_json"]),
                                     separators=(',', ':'))
                taxon   = taxon_from_path(row["image_path"])
                self.samples.append({"image": img, "caption_json": cap_str,
                                     "taxon": taxon})
            except Exception as e:
                print(f"Error loading sample {row.get('image_path', 'unknown')}: {e}")

        if shuffle:
            random.shuffle(self.samples)
        aug_label = "augmentation ON" if augment else "augmentation OFF"
        print(f"Done loading images into RAM ({aug_label}).", flush=True)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        if self.transforms is not None:
            sample = {**sample, "image": self.transforms(sample["image"])}
        return self.formatter(sample)

class LazyDiskDataset(torch.utils.data.Dataset):
    """Drop-in replacement for InMemoryDataset that decodes images on demand.
    Holds only paths/captions/taxa in RAM (~MBs); reads+decodes JPEG per item.
    Worker-safe — fork footprint is trivial, so num_workers > 0 is fine here."""
    def __init__(self, hf_dataset, formatter, shuffle: bool = False,
                 augment: bool = False):
        self.formatter  = formatter
        self.transforms = TRAIN_TRANSFORMS if augment else None

        print(f"Indexing {len(hf_dataset)} samples (lazy load)...", flush=True)
        self.samples = []
        n_missing = 0
        for row in hf_dataset:
            img_path = DATA_ROOT / row["image_path"]
            if not img_path.exists():           # cheap up-front validation, no decode
                n_missing += 1
                print(f"Missing image, skipping: {img_path}")
                continue
            cap_str = json.dumps(json.loads(row["caption_json"]),
                                 separators=(',', ':'))
            taxon   = taxon_from_path(row["image_path"])
            self.samples.append({"path": img_path, "caption_json": cap_str,
                                 "taxon": taxon})

        if shuffle:
            random.shuffle(self.samples)
        aug_label = "augmentation ON" if augment else "augmentation OFF"
        print(f"Indexed {len(self.samples)} samples "
              f"({n_missing} missing) ({aug_label}).", flush=True)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")     # decode on demand
        if self.transforms is not None:
            img = self.transforms(img)
        return self.formatter({"image": img,
                               "caption_json": s["caption_json"],
                               "taxon": s["taxon"]})

# 6. Load Raw Datasets from Disk
print("Loading datasets from disk...")
raw_train = datasets.load_from_disk(str(DATA_DIR))
raw_val = datasets.load_from_disk(str(VAL_DIR)) if VAL_DIR.exists() else None

print(f"Train samples: {len(raw_train)}")
print(f"Val samples:   {len(raw_val) if raw_val else 0}")
print("Model and raw datasets ready for training.")

Loading datasets from disk...
Train samples: 4819
Val samples:   964
Model and raw datasets ready for training.


In [3]:
# # =============================================================================
# # CELL 3: P1 Training
# # =============================================================================
# # 1. Configure Directories and datasets
# P1_OUTPUT_DIR = OUTPUT_DIR / "phase_1"
# P1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# P1_LOGGING_DIR = LOGGING_DIR / "phase_1"
# P1_LOGGING_DIR.mkdir(parents=True, exist_ok=True)

# print("Converting to in-memory datasets...")
# train_dataset = LazyDiskDataset(raw_train, format_phase1, shuffle=True, augment=True)
# val_dataset = LazyDiskDataset(raw_val, format_phase1, shuffle=True) if raw_val else None

# # 2. Define SFTConfig for Full Fine-Tuning
# training_args = SFTConfig(
#     output_dir              = str(P1_OUTPUT_DIR),
#     run_name                = "qwen3-vl-full-ft-phase1",
    
#     # Data Loading
#     per_device_train_batch_size  = BATCH_SIZE,
#     per_device_eval_batch_size   = max(1, BATCH_SIZE // 2),
#     gradient_accumulation_steps  = GRAD_ACCUM,
#     dataloader_num_workers       = 3,
#     dataloader_pin_memory        = True,
    
#     # Training Hyperparameters
#     num_train_epochs             = P1_NUM_EPOCHS,
#     learning_rate                = P1_LEARNING_RATE,
#     warmup_steps                 = P1_WARMUP_STEPS,
#     weight_decay                 = WEIGHT_DECAY,
#     lr_scheduler_type            = SCHEDULER_TYPE,
    
#     # Precision and Optimization
#     bf16                         = True,
#     fp16                         = False,
#     optim                        = OPTIM,
    
#     # Logging and Saving
#     logging_steps                = 10,
#     logging_dir                  = str(P1_LOGGING_DIR),
#     report_to                    = "tensorboard", 
    
#     # Evaluation and Checkpointing
#     eval_strategy                = "epoch" if raw_val else "no",
#     save_strategy                = "best",
#     save_total_limit             = 1,
#     load_best_model_at_end      = True,
#     metric_for_best_model       = "eval_loss",
    
#     # Miscellaneous
#     seed                         = 42,
#     max_seq_length               = MAX_SEQ_LEN,
#     remove_unused_columns        = False,
# )

# # 3. Initialize Data Collator
# data_collator = UnslothVisionDataCollator(
#     model,
#     tokenizer,
#     train_on_responses_only = True,
#     instruction_part        = "user\n",
#     response_part           = "assistant\n",
# )

# # 4. Initialize SFTTrainer
# print("Initializing Trainer...")
# trainer = SFTTrainer(
#     model         = model,
#     tokenizer     = tokenizer,
#     args          = training_args,
#     train_dataset = train_dataset,
#     eval_dataset  = val_dataset,
#     data_collator = data_collator,
# )

# # 5. Print Memory Usage Before Training
# if torch.cuda.is_available():
#     mem_before = torch.cuda.max_memory_reserved() / 1e9
#     print(f"Reserved Memory Before Training: {mem_before:.2f} GB")

# # =============================================================================
# # Execute Training Loop
# # =============================================================================

# # 6. Update the trainer args to reflect the current run length
# # This ensures we don't accidentally run for 3 epochs if we only wanted 1.

# # 7. Execute Training
# print(f"Starting training for {P1_NUM_EPOCHS} epoch(s)...")
# print("-" * 50)

# try:
#     train_result = trainer.train()
    
#     # Post-Training Metrics
#     print("-" * 50)
#     print("Training Complete.")
#     print(f"Final Training Loss: {train_result.training_loss:.4f}")
    
#     # Memory Usage After Training
#     if torch.cuda.is_available():
#         mem_after = torch.cuda.max_memory_reserved() / 1e9
#         print(f"Peak Reserved Memory: {mem_after:.2f} GB")
#         print(f"Memory Used by Model/Training: ~{mem_after - 26.66:.2f} GB") # Subtracting initial load estimate
        
#     # Save the Final Checkpoint
#     final_checkpoint_dir = P1_OUTPUT_DIR / "best"
#     final_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
#     print(f"Saving final model to: {final_checkpoint_dir}")
#     trainer.save_model(str(final_checkpoint_dir))
    
#     # Save tokenizer as well for completeness
#     #tokenizer.save_pretrained(str(final_checkpoint_dir))
    
#     print("Phase 1, training with caption instructions done.")

# except Exception as e:
#     print(f"Training failed with error: {e}")
#     import traceback
#     traceback.print_exc()

In [4]:
# =============================================================================
# CELL 4: P2 Training (Fine-Tuning on P1 Best Checkpoint)
# =============================================================================

# 1. Configure Directories and datasets
P2_OUTPUT_DIR = OUTPUT_DIR / "phase_2"
P2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
P2_LOGGING_DIR = LOGGING_DIR / "phase_2"
P2_LOGGING_DIR.mkdir(parents=True, exist_ok=True)

P1_OUTPUT_DIR = OUTPUT_DIR / "phase_1"
P1_BEST = P1_OUTPUT_DIR / "best"
# 2. Load the Best Model from Phase 1
if not P1_BEST.exists():
    raise FileNotFoundError(f"Phase 1 best checkpoint not found at {P1_BEST}. Ensure Phase 1 completed successfully.")

print("Converting to in-memory datasets...")
train_dataset = LazyDiskDataset(raw_train, format_phase2, shuffle=True, augment=True)
val_dataset = LazyDiskDataset(raw_val, format_phase2, shuffle=True) if raw_val else None

print(f"Loading best checkpoint from Phase 1: {P1_BEST}")
print("This may take a moment...")

# Reload the model with the weights from the best P1 checkpoint
# Note: We must reload with full_finetuning=True and same dtype to match P1 config
model, tokenizer = FastVisionModel.from_pretrained(
    model_name      = str(P1_BEST),
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = False,          # Full FT
    dtype           = DTYPE,          # bfloat16
    full_finetuning = True,
)

# Verify parameters are trainable
trainable_params_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params_p2 = sum(p.numel() for p in model.parameters())
print(f"Phase 2 Model - Trainable parameters: {trainable_params_p2:,} / {total_params_p2:,} ({100*trainable_params_p2/total_params_p2:.2f}%)")

# 3. Configure SFTConfig for Phase 2
training_args = SFTConfig(
    output_dir              = str(P2_OUTPUT_DIR),
    run_name                = "qwen3-vl-full-ft-phase2",
    
    # Data Loading
    per_device_train_batch_size  = BATCH_SIZE,
    per_device_eval_batch_size   = max(1, BATCH_SIZE // 2),
    gradient_accumulation_steps  = GRAD_ACCUM,
    dataloader_num_workers       = 3,
    dataloader_pin_memory        = True,
    
    # Training Hyperparameters - Phase 2 Specifics
    num_train_epochs             = P2_NUM_EPOCHS,   # Typically higher than P1
    learning_rate                = P2_LEARNING_RATE, # Typically lower than P1
    warmup_steps                 = P2_WARMUP_STEPS,
    weight_decay                 = WEIGHT_DECAY,
    lr_scheduler_type            = SCHEDULER_TYPE,
    
    # Precision and Optimization
    bf16                         = True,
    fp16                         = False,
    optim                        = OPTIM,
    
    # Logging and Saving
    logging_steps                = 10,
    logging_dir                  = str(P2_LOGGING_DIR),
    report_to                    = "tensorboard", 
    
    # Evaluation and Checkpointing
    eval_strategy                = "epoch" if raw_val else "no",
    save_strategy                = "best",
    save_total_limit             = 1,
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
    
    # Miscellaneous
    seed                         = 42,
    max_seq_length               = MAX_SEQ_LEN,
    remove_unused_columns        = False,
)

# 4. Initialize Data Collator for Phase 2
# Assuming same formatting requirements as Phase 1
data_collator = UnslothVisionDataCollator(
    model,
    tokenizer,
    train_on_responses_only = True,
    instruction_part        = "user\n",
    response_part           = "assistant\n",
)

# 5. Initialize SFTTrainer for Phase 2
print("Initializing Phase 2 Trainer...")
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    args          = training_args,
    train_dataset = train_dataset, # Re-use the same in-memory dataset
    eval_dataset  = val_dataset,   # Re-use the same validation set
    data_collator = data_collator,
)

# 6. Print Memory Usage Before Phase 2
if torch.cuda.is_available():
    mem_before = torch.cuda.max_memory_reserved() / 1e9
    print(f"Reserved Memory Before Phase 2 Training: {mem_before:.2f} GB")

# =============================================================================
# Execute Phase 2 Training Loop
# =============================================================================

print(f"Starting Phase 2 training for {P2_NUM_EPOCHS} epoch(s)...")
print("-" * 50)

try:
    train_result = trainer.train()
    
    # Post-Training Metrics
    print("-" * 50)
    print("Phase 2 Training Complete.")
    print(f"Final Training Loss: {train_result.training_loss:.4f}")
    
    # Memory Usage After Phase 2
    if torch.cuda.is_available():
        mem_after = torch.cuda.max_memory_reserved() / 1e9
        print(f"Peak Reserved Memory (Phase 2): {mem_after:.2f} GB")
        
    # Save the Final Phase 2 Checkpoint
    final_checkpoint_dir = P2_OUTPUT_DIR / "best"
    final_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Saving final Phase 2 model to: {final_checkpoint_dir}")
    trainer.save_model(str(final_checkpoint_dir))
    
    # Save tokenizer as well for completeness
    #tokenizer.save_pretrained(str(final_checkpoint_dir))
    
    print("Phase 2 fine-tuning complete.")

except Exception as e:
    print(f"Phase 2 Training failed with error: {e}")
    import traceback
    traceback.print_exc()


Converting to in-memory datasets...
Indexing 4819 samples (lazy load)...
Indexed 4819 samples (0 missing) (augmentation ON).
Indexing 964 samples (lazy load)...
Indexed 964 samples (0 missing) (augmentation OFF).
Loading best checkpoint from Phase 1: /scratch/cpowel21/vllm_tuning/checkpoints/phase_1/best
This may take a moment...
==((====))==  Unsloth 2026.5.8: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Phase 2 Model - Trainable parameters: 4,437,815,808 / 4,437,815,808 (100.00%)
Unsloth: Model does not have a default image size - using 512
Initializing Phase 2 Trainer...


[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


...

Unsloth: Restored added_tokens_decoder metadata in /scratch/cpowel21/vllm_tuning/checkpoints/phase_2/checkpoint-804/tokenizer_config.json.
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


--------------------------------------------------
Phase 2 Training Complete.
Final Training Loss: 0.0308
Peak Reserved Memory (Phase 2): 46.07 GB
Saving final Phase 2 model to: /scratch/cpowel21/vllm_tuning/checkpoints/phase_2/best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /scratch/cpowel21/vllm_tuning/checkpoints/phase_2/best/tokenizer_config.json.


Phase 2 fine-tuning complete.


In [5]:
# =============================================================================
# CELL: Phase 2 Evaluation — Best Checkpoint vs Validation Set
# =============================================================================
# Evaluates the phase 2 best checkpoint (image + taxon, no system prompt)
# against the full validation set. Reports raw and lenient JSON validity,
# exact match, and per-field presence / accuracy across all 16 v2 schema fields.
# Outputs: printed table + eval_p2_results.csv

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_BATCH_SIZE = 8
MAX_GEN_TOKENS  = 300
P2_BEST         = P2_OUTPUT_DIR / "best"
EVAL_CSV        = SCRATCH_ROOT / "eval_p2_results.csv"

# All leaf fields to evaluate (v2 simplified-phenology schema, 16 fields)
FIELDS = [
    "type",
    "attached_photo",
    "structures.foliage",
    "structures.foliage_type",
    "structures.stem",
    "structures.phenology.flower",
    "structures.phenology.fruit",
    "structures.phenology.pollen_cone",
    "structures.phenology.seed_cone",
    "structures.phenology.sporulating",
    "structures.phenology.reproductive_unknown",
    "refs.label",
    "refs.barcode",
    "refs.stamp",
    "refs.crc",
    "refs.scale_bar",
]

# ── Load model ────────────────────────────────────────────────────────────────
if not P2_BEST.exists():
    raise FileNotFoundError(f"Phase 2 checkpoint not found: {P2_BEST}")

print(f"Loading: {P2_BEST}")
torch.cuda.empty_cache()

eval_model, eval_tokenizer = FastVisionModel.from_pretrained(
    model_name      = str(P2_BEST),
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = False,
    dtype           = DTYPE,
    full_finetuning = True,     # must match training config
)
FastVisionModel.for_inference(eval_model)   # enable optimised inference path
eval_device = next(eval_model.parameters()).device
print(f"Loaded on {eval_device}.\n")

# ── Load validation set ───────────────────────────────────────────────────────
# Re-uses raw_val loaded in Cell 2; loads images fresh for inference.
# Stores taxon per sample to mirror the Phase-2 (image + taxon) input format.
print("Loading validation samples...")
eval_samples = []
for row in raw_val:
    try:
        img   = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
        taxon = Path(row["image_path"]).parent.name.replace("_", " ")
        eval_samples.append((img, row["caption_json"].strip(), taxon))
    except Exception as e:
        print(f"  Skipping {row.get('image_path', '?')}: {e}")
print(f"  {len(eval_samples)} samples ready.\n")

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_nested_value(d, key_path):
    """Dot-notation access into nested dict (reused from eval helpers)."""
    for k in key_path.split("."):
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return None
    return d

def parse_strict(text):
    """No cleaning beyond strip. Returns (dict|None, bool:is_raw_valid)."""
    s = text.strip()
    is_raw_valid = s.startswith("{") and s.endswith("}")
    try:
        return json.loads(s), is_raw_valid
    except json.JSONDecodeError:
        return None, is_raw_valid

def parse_lenient(text):
    """Strip markdown fence only, then find outermost braces."""
    s = text.strip()
    if "```" in s:
        parts = s.split("```")
        s = parts[1].strip() if len(parts) > 1 else s
        if s.startswith("json"):
            s = s[4:].strip()
    start, end = s.find("{"), s.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(s[start:end + 1])
    except json.JSONDecodeError:
        return None

# ── Evaluation loop ───────────────────────────────────────────────────────────
n_total         = len(eval_samples)
n_raw_valid     = 0   # parseable with no cleaning
n_lenient_valid = 0   # parseable after minimal fence removal
n_exact_match   = 0   # normalised via re-serialisation

field_present = {f: 0 for f in FIELDS}
field_correct = {f: 0 for f in FIELDS}

print(f"Running inference on {n_total} samples (batch size = {EVAL_BATCH_SIZE})...")

for i in tqdm(range(0, n_total, EVAL_BATCH_SIZE), desc="Evaluating"):
    batch                     = eval_samples[i : i + EVAL_BATCH_SIZE]
    batch_imgs, batch_gts, batch_taxa = zip(*batch)

    # Phase 2 format: image + taxon, no system prompt
    texts = [
        eval_tokenizer.apply_chat_template(
            [{"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": f"Taxon: {taxon}"},
            ]}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for img, taxon in zip(batch_imgs, batch_taxa)
    ]

    inputs = eval_tokenizer(
        text              = texts,
        images            = list(batch_imgs),
        return_tensors    = "pt",
        padding           = True,
        truncation        = True,
        max_length        = MAX_SEQ_LEN,
    ).to(eval_device)

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens = MAX_GEN_TOKENS,
            do_sample      = False,         # greedy; deterministic eval
            use_cache      = True,
            pad_token_id   = eval_tokenizer.pad_token_id or eval_tokenizer.eos_token_id,
        )

    # Slice to new tokens only — avoids prompt-stripping errors from vision tokens
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    responses  = eval_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for response, gt_str in zip(responses, batch_gts):
        try:
            gt = json.loads(gt_str)
        except json.JSONDecodeError:
            continue

        _, is_raw_valid  = parse_strict(response)
        pred             = parse_lenient(response)

        if is_raw_valid:
            n_raw_valid += 1
        if pred is not None:
            n_lenient_valid += 1
        else:
            continue

        if json.dumps(pred, sort_keys=True) == json.dumps(gt, sort_keys=True):
            n_exact_match += 1

        for field in FIELDS:
            pred_val = get_nested_value(pred, field)
            gt_val   = get_nested_value(gt,   field)
            if pred_val is not None:
                field_present[field] += 1
            if pred_val == gt_val:
                field_correct[field] += 1

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 2 EVALUATION RESULTS")
print("=" * 60)
print(f"  Samples evaluated   : {n_total}")
print(f"  Raw JSON valid      : {n_raw_valid:4d}  ({100 * n_raw_valid / n_total:.1f}%)")
print(f"  Lenient JSON valid  : {n_lenient_valid:4d}  ({100 * n_lenient_valid / n_total:.1f}%)")
print(f"  Exact match         : {n_exact_match:4d}  ({100 * n_exact_match / n_total:.1f}%)")
print()
print(f"  {'Field':<40} {'present%':>9} {'correct%':>9}")
print("  " + "-" * 58)
for field in FIELDS:
    pct_p = 100 * field_present[field] / n_lenient_valid if n_lenient_valid else float("nan")
    pct_c = 100 * field_correct[field] / n_lenient_valid if n_lenient_valid else float("nan")
    print(f"  {field:<40} {pct_p:8.1f} {pct_c:8.1f}")
print("=" * 60)

# ── Save CSV ──────────────────────────────────────────────────────────────────
summary_row = {
    "field":            "SUMMARY",
    "n_total":          n_total,
    "raw_valid_%":      round(100 * n_raw_valid     / n_total, 1),
    "lenient_valid_%":  round(100 * n_lenient_valid / n_total, 1),
    "exact_match_%":    round(100 * n_exact_match   / n_total, 1),
    "present_%":        "",
    "correct_%":        "",
}
field_rows = [
    {
        "field":           f,
        "n_total":         "",
        "raw_valid_%":     "",
        "lenient_valid_%": "",
        "exact_match_%":   "",
        "present_%":       round(100 * field_present[f] / n_lenient_valid, 1) if n_lenient_valid else None,
        "correct_%":       round(100 * field_correct[f] / n_lenient_valid, 1) if n_lenient_valid else None,
    }
    for f in FIELDS
]

with open(EVAL_CSV, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=summary_row.keys())
    writer.writeheader()
    writer.writerow(summary_row)
    writer.writerows(field_rows)

print(f"Results saved: {EVAL_CSV}")

Loading: /scratch/cpowel21/vllm_tuning/checkpoints/phase_2/best
==((====))==  Unsloth 2026.5.8: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Loaded on cuda:0.

Loading validation samples...
  964 samples ready.

Running inference on 964 samples (batch size = 8)...


Evaluating: 100%|██████████| 121/121 [11:30<00:00,  5.71s/it]


PHASE 2 EVALUATION RESULTS
  Samples evaluated   : 964
  Raw JSON valid      :  964  (100.0%)
  Lenient JSON valid  :  964  (100.0%)
  Exact match         :  497  (51.6%)

  Field                                     present%  correct%
  ----------------------------------------------------------
  type                                        100.0    100.0
  attached_photo                              100.0     99.3
  structures.foliage                          100.0     97.9
  structures.foliage_type                     100.0     92.9
  structures.stem                             100.0     87.9
  structures.phenology.flower                 100.0     83.5
  structures.phenology.fruit                  100.0     82.5
  structures.phenology.pollen_cone            100.0     98.8
  structures.phenology.seed_cone              100.0     98.9
  structures.phenology.sporulating            100.0     97.2
  structures.phenology.reproductive_unknown    100.0     99.7
  refs.label                   

In [9]:
# =============================================================================
# CELL: Held-Out TEST Evaluation — Phase 2 Best Checkpoint vs Test Set
# =============================================================================
# Self-contained. Loads its own model, helpers, and the test split from disk.
# Mirrors the Phase-2 input format (image + taxon, no system prompt).
# RUN ONCE, at the very end — the test set is held out and should be touched
# rarely. Reports raw/lenient JSON validity, exact match, and per-field
# presence / accuracy across all 16 v2 schema fields.
# Outputs: printed table + eval_test_results.csv

import csv
import json
from pathlib import Path
import torch
from tqdm import tqdm
from PIL import Image
from unsloth import FastVisionModel
import datasets

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_BATCH_SIZE = 8
MAX_GEN_TOKENS  = 300
P2_BEST         = OUTPUT_DIR / "phase_2" / "best"
TEST_CSV        = SCRATCH_ROOT / "eval_test_results.csv"

# All leaf fields to evaluate (v2 simplified-phenology schema, 16 fields)
FIELDS = [
    "type",
    "attached_photo",
    "structures.foliage",
    "structures.foliage_type",
    "structures.stem",
    "structures.phenology.flower",
    "structures.phenology.fruit",
    "structures.phenology.pollen_cone",
    "structures.phenology.seed_cone",
    "structures.phenology.sporulating",
    "structures.phenology.reproductive_unknown",
    "refs.label",
    "refs.barcode",
    "refs.stamp",
    "refs.crc",
    "refs.scale_bar",
]

# ── Load test set from disk (self-contained) ──────────────────────────────────
if not TEST_DIR.exists():
    raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")

print(f"Loading test split: {TEST_DIR}")
raw_test = datasets.load_from_disk(str(TEST_DIR))
print(f"  {len(raw_test)} rows.")

# ── Load model ────────────────────────────────────────────────────────────────
if not P2_BEST.exists():
    raise FileNotFoundError(f"Phase 2 checkpoint not found: {P2_BEST}")

print(f"Loading: {P2_BEST}")
torch.cuda.empty_cache()

test_model, test_tokenizer = FastVisionModel.from_pretrained(
    model_name      = str(P2_BEST),
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = False,
    dtype           = DTYPE,
    full_finetuning = True,     # must match training config
)
FastVisionModel.for_inference(test_model)
test_device = next(test_model.parameters()).device
print(f"Loaded on {test_device}.\n")

# ── Load test samples (image + taxon, mirrors Phase-2 format) ─────────────────
print("Loading test samples...")
test_samples = []
for row in raw_test:
    try:
        img   = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
        taxon = Path(row["image_path"]).parent.name.replace("_", " ")
        test_samples.append((img, row["caption_json"].strip(), taxon))
    except Exception as e:
        print(f"  Skipping {row.get('image_path', '?')}: {e}")
print(f"  {len(test_samples)} samples ready.\n")

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_nested_value(d, key_path):
    """Dot-notation access into nested dict."""
    for k in key_path.split("."):
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return None
    return d

def parse_strict(text):
    """No cleaning beyond strip. Returns (dict|None, bool:is_raw_valid)."""
    s = text.strip()
    is_raw_valid = s.startswith("{") and s.endswith("}")
    try:
        return json.loads(s), is_raw_valid
    except json.JSONDecodeError:
        return None, is_raw_valid

def parse_lenient(text):
    """Strip markdown fence only, then find outermost braces."""
    s = text.strip()
    if "```" in s:
        parts = s.split("```")
        s = parts[1].strip() if len(parts) > 1 else s
        if s.startswith("json"):
            s = s[4:].strip()
    start, end = s.find("{"), s.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(s[start:end + 1])
    except json.JSONDecodeError:
        return None

# ── Evaluation loop ───────────────────────────────────────────────────────────
n_total         = len(test_samples)
n_raw_valid     = 0
n_lenient_valid = 0
n_exact_match   = 0

field_present = {f: 0 for f in FIELDS}
field_correct = {f: 0 for f in FIELDS}

print(f"Running inference on {n_total} samples (batch size = {EVAL_BATCH_SIZE})...")

for i in tqdm(range(0, n_total, EVAL_BATCH_SIZE), desc="Testing"):
    batch                             = test_samples[i : i + EVAL_BATCH_SIZE]
    batch_imgs, batch_gts, batch_taxa = zip(*batch)

    texts = [
        test_tokenizer.apply_chat_template(
            [{"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": f"Taxon: {taxon}"},
            ]}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for img, taxon in zip(batch_imgs, batch_taxa)
    ]

    inputs = test_tokenizer(
        text              = texts,
        images            = list(batch_imgs),
        return_tensors    = "pt",
        padding           = True,
        truncation        = True,
        max_length        = MAX_SEQ_LEN,
    ).to(test_device)

    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens = MAX_GEN_TOKENS,
            do_sample      = False,
            use_cache      = True,
            pad_token_id   = test_tokenizer.pad_token_id or test_tokenizer.eos_token_id,
        )

    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    responses  = test_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for response, gt_str in zip(responses, batch_gts):
        try:
            gt = json.loads(gt_str)
        except json.JSONDecodeError:
            continue

        _, is_raw_valid = parse_strict(response)
        pred            = parse_lenient(response)

        if is_raw_valid:
            n_raw_valid += 1
        if pred is not None:
            n_lenient_valid += 1
        else:
            continue

        if json.dumps(pred, sort_keys=True) == json.dumps(gt, sort_keys=True):
            n_exact_match += 1

        for field in FIELDS:
            pred_val = get_nested_value(pred, field)
            gt_val   = get_nested_value(gt,   field)
            if pred_val is not None:
                field_present[field] += 1
            if pred_val == gt_val:
                field_correct[field] += 1

# ── Report ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("HELD-OUT TEST EVALUATION RESULTS")
print("=" * 60)
print(f"  Samples evaluated   : {n_total}")
print(f"  Raw JSON valid      : {n_raw_valid:4d}  ({100 * n_raw_valid / n_total:.1f}%)")
print(f"  Lenient JSON valid  : {n_lenient_valid:4d}  ({100 * n_lenient_valid / n_total:.1f}%)")
print(f"  Exact match         : {n_exact_match:4d}  ({100 * n_exact_match / n_total:.1f}%)")
print()
print(f"  {'Field':<40} {'present%':>9} {'correct%':>9}")
print("  " + "-" * 58)
for field in FIELDS:
    pct_p = 100 * field_present[field] / n_lenient_valid if n_lenient_valid else float("nan")
    pct_c = 100 * field_correct[field] / n_lenient_valid if n_lenient_valid else float("nan")
    print(f"  {field:<40} {pct_p:8.1f} {pct_c:8.1f}")
print("=" * 60)

# ── Save CSV ──────────────────────────────────────────────────────────────────
summary_row = {
    "field":            "SUMMARY",
    "n_total":          n_total,
    "raw_valid_%":      round(100 * n_raw_valid     / n_total, 1),
    "lenient_valid_%":  round(100 * n_lenient_valid / n_total, 1),
    "exact_match_%":    round(100 * n_exact_match   / n_total, 1),
    "present_%":        "",
    "correct_%":        "",
}
field_rows = [
    {
        "field":           f,
        "n_total":         "",
        "raw_valid_%":     "",
        "lenient_valid_%": "",
        "exact_match_%":   "",
        "present_%":       round(100 * field_present[f] / n_lenient_valid, 1) if n_lenient_valid else None,
        "correct_%":       round(100 * field_correct[f] / n_lenient_valid, 1) if n_lenient_valid else None,
    }
    for f in FIELDS
]

with open(TEST_CSV, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=summary_row.keys())
    writer.writeheader()
    writer.writerow(summary_row)
    writer.writerows(field_rows)

print(f"Results saved: {TEST_CSV}")

Loading test split: /scratch/cpowel21/vllm_tuning/data/test
  643 rows.
Loading: /scratch/cpowel21/vllm_tuning/checkpoints/phase_2/best
==((====))==  Unsloth 2026.5.8: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Loaded on cuda:0.

Loading test samples...
  643 samples ready.

Running inference on 643 samples (batch size = 8)...


Testing: 100%|██████████| 81/81 [07:13<00:00,  5.35s/it]


HELD-OUT TEST EVALUATION RESULTS
  Samples evaluated   : 643
  Raw JSON valid      :  643  (100.0%)
  Lenient JSON valid  :  643  (100.0%)
  Exact match         :  321  (49.9%)

  Field                                     present%  correct%
  ----------------------------------------------------------
  type                                        100.0    100.0
  attached_photo                              100.0     99.5
  structures.foliage                          100.0     97.8
  structures.foliage_type                     100.0     92.2
  structures.stem                             100.0     88.6
  structures.phenology.flower                 100.0     80.7
  structures.phenology.fruit                  100.0     79.5
  structures.phenology.pollen_cone            100.0     99.5
  structures.phenology.seed_cone              100.0     99.8
  structures.phenology.sporulating            100.0     97.0
  structures.phenology.reproductive_unknown    100.0     99.2
  refs.label             

In [7]:
# =============================================================================
# CELL 5: Export to GGUF (F16 + Q8_0 + mmproj)
# =============================================================================
# # Hackey method to get conversion on HPC without compilation. Retained for clarity.
# import subprocess, sys, os
# from pathlib import Path

# LLAMA_CPP_DIR = SCRATCH_ROOT / "llama_cpp_repo"

# if not LLAMA_CPP_DIR.exists():
#     subprocess.run([
#         "git", "clone", "--depth", "1",
#         "https://github.com/ggml-org/llama.cpp",
#         str(LLAMA_CPP_DIR)
#     ], check=True)

# CONVERTER_PATH = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"

# # Install the gguf package from the cloned repo (pure Python, no compile)
# subprocess.run([
#     sys.executable, "-m", "pip", "install",
#     str(LLAMA_CPP_DIR / "gguf-py"),
#     "--quiet"
# ], check=True)


# Requires llama_cpp_repo cloned and gguf-py installed (see commented block)
#   1. Main model  → F16  (reference quality)
#   2. Main model  → Q8_0 (compressed; llama-quantize unavailable on HPC)
#   3. mmproj      → F16  (single projector shared by both quants)

P2_BEST_CHECKPOINT = SCRATCH_ROOT / "checkpoints" / "phase_2" / "best"
GGUF_OUTPUT_DIR    = SCRATCH_ROOT / "gguf_output"
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GGUF_F16    = GGUF_OUTPUT_DIR / "qwen3-vl-4b-herbarium-f16.gguf"
GGUF_Q8     = GGUF_OUTPUT_DIR / "qwen3-vl-4b-herbarium-q8.gguf"
GGUF_MMPROJ = GGUF_OUTPUT_DIR / "mmproj-qwen3-vl-4b-herbarium-f16.gguf"

LLAMA_CPP_DIR  = SCRATCH_ROOT / "llama_cpp_repo"
CONVERTER_PATH = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"

if not CONVERTER_PATH.exists():
    raise FileNotFoundError(f"Converter not found: {CONVERTER_PATH}")
if not P2_BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {P2_BEST_CHECKPOINT}")

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{Path(sys.prefix) / 'lib'}:{env.get('LD_LIBRARY_PATH', '')}"
common_args = dict(check=True, capture_output=True, text=True,
                   cwd=str(LLAMA_CPP_DIR), env=env)

# --- F16 ---
print("Converting F16...")
result = subprocess.run(
    [sys.executable, str(CONVERTER_PATH),
     str(P2_BEST_CHECKPOINT),
     "--outfile", str(GGUF_F16),
     "--outtype", "f16"],
    **common_args
)
print(result.stdout)
print(f"F16:    {GGUF_F16.stat().st_size / 1e9:.2f} GB")

# --- Q8_0 ---
print("Converting Q8_0...")
result = subprocess.run(
    [sys.executable, str(CONVERTER_PATH),
     str(P2_BEST_CHECKPOINT),
     "--outfile", str(GGUF_Q8),
     "--outtype", "q8_0"],
    **common_args
)
print(result.stdout)
print(f"Q8_0:   {GGUF_Q8.stat().st_size / 1e9:.2f} GB")

# --- mmproj (F16; one projector serves all quants) ---
print("Converting mmproj...")
result = subprocess.run(
    [sys.executable, str(CONVERTER_PATH),
     str(P2_BEST_CHECKPOINT),
     "--outfile", str(GGUF_MMPROJ),
     "--outtype", "f16",
     "--mmproj"],
    **common_args
)
print(result.stdout)
print(f"mmproj: {GGUF_MMPROJ.stat().st_size / 1e9:.2f} GB")

# --- Verify all outputs present ---
print("\nOutput files:")
for p in [GGUF_F16, GGUF_Q8, GGUF_MMPROJ]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p.name}")

Converting F16...

F16:    8.05 GB
Converting Q8_0...

Q8_0:   4.28 GB
Converting mmproj...

mmproj: 0.84 GB

Output files:
  [OK] qwen3-vl-4b-herbarium-f16.gguf
  [OK] qwen3-vl-4b-herbarium-q8.gguf
  [OK] mmproj-qwen3-vl-4b-herbarium-f16.gguf


In [8]:
# # =============================================================================
# # DIAGNOSTIC: Extract & save validation GBIF IDs to .txt
# # =============================================================================

# from pathlib import Path
# import datasets

# VAL_DIR = Path("/scratch/cpowel21/vllm_tuning/data/val")
# OUTPUT_FILE = Path("/scratch/cpowel21/vllm_tuning/val_gbif_ids.txt")

# if not VAL_DIR.exists():
#     print(f"❌ Validation directory not found: {VAL_DIR}")
# else:
#     # Load dataset & extract stems
#     val_ds = datasets.load_from_disk(str(VAL_DIR))
#     gbif_ids = sorted(set(Path(img_path).stem for img_path in val_ds["image_path"]))
    
#     # Save to txt
#     OUTPUT_FILE.write_text("\n".join(gbif_ids))
    
#     print(f"✅ Saved {len(gbif_ids)} unique GBIF IDs to: {OUTPUT_FILE}")
#     print("\n🔍 Preview (first 10):")
#     for gbif_id in gbif_ids[:10]:
#         print(gbif_id)
#     print(f"... and {len(gbif_ids) - 10} more")
